In [223]:
import numpy as np


def structure(value):
    def decorator(func):
        func.structure = value
        return func

    return decorator


def changes(l, rng=None):
    rng = rng or np.random.default_rng()
    changes = []
    for _ in range(l):
        if rng.random() < 0.1:
            n = int(rng.integers(50, 1000))
            P = int(rng.integers(0, 3))
        else:
            n = int(rng.integers(10, 60))
            P = int(rng.integers(0, 5))
        positives = np.arange(P)
        changes.append((positives, n))
    return changes


def predictions(p, n, mu=None, reps=1, rng=None):
    rng = rng or np.random.default_rng()
    if mu is None:
        mu = np.linspace(0.0, 4.0, 40)

    p = np.asarray(p)
    if p.dtype == bool:
        pos = p
    else:
        pos = np.isin(np.arange(n), p.astype(int))

    mus = np.repeat(mu, reps)
    base = rng.standard_normal(n)
    scores = np.tile(base, (mus.size, 1))
    scores[:, pos] += mus[:, None]
    return scores


def topk(order, k):
    pred = np.zeros(order.shape, bool)
    if k > 0:
        pred[np.arange(order.shape[0])[:, None], order[:, :k]] = True
    return pred


def threshold(metric, S, y):
    order = np.argsort(-S, axis=1)
    n_mu, n = S.shape
    vals = np.column_stack([metric(topk(order, k), y) for k in range(n + 1)])
    return vals.argmax(axis=1)


def score(metric, S, kstar, y):
    order = np.argsort(-S, axis=1)
    pred = np.zeros(S.shape, bool)
    for r, k in enumerate(kstar):
        if k > 0:
            pred[r, order[r, :k]] = True
    return metric(pred, y)


@structure("label")
def f1(pred, y):
    TP = (pred & y).sum(axis=1)
    FP = (pred & ~y).sum(axis=1)
    FN = (~pred & y).sum(axis=1)
    den = 2 * TP + FP + FN
    return np.where(den > 0, 2 * TP / np.maximum(den, 1), 1.0)


@structure("label")
def mcc(pred, y):
    TP = (pred & y).sum(axis=1).astype(float)
    FP = (pred & ~y).sum(axis=1).astype(float)
    FN = (~pred & y).sum(axis=1).astype(float)
    TN = (~pred & ~y).sum(axis=1).astype(float)
    num = TP * TN - FP * FN
    den = np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    return np.where(den > 0, num / np.maximum(den, 1e-12), 0.0)


@structure("rank")
def mrr(S, y):
    order = np.argsort(-S, axis=1)  # node indices by descending score, per mu
    ranked_pos = y[order]  # is the node at this rank a positive
    has_pos = ranked_pos.any(axis=1)
    first = np.argmax(ranked_pos, axis=1)  # rank-1 index of the first positive
    return np.where(has_pos, 1.0 / (first + 1), 0.0)


def theta_star(
    C, mu, metric=f1, M=300, alpha=(1, 1), beta=(1, 1), delta=(1, 1), rng=None
):
    rng = rng or np.random.default_rng()
    S = [predictions(p, n, mu, rng=rng) for p, n in C]
    Y = [np.isin(np.arange(n), list(p)) for p, n in C]

    thresholds = (
        [None] * len(C)
        if metric.structure != "label"
        else [threshold(metric, s, y) for s, y in zip(S, Y)]
    )
    level = np.mean(
        [
            (
                metric(S[i], Y[i])
                if metric.structure != "label"
                else score(metric, S[i], thresholds[i], Y[i])
            )
            for i in range(len(C))
        ],
        axis=0,
    )

    agg = np.empty((M, mu.size))
    for m in range(M):
        a = rng.beta(*alpha)
        b = rng.beta(*beta)
        d = rng.beta(*delta)
        per_change = np.empty((len(C), mu.size))
        for i, (s, y) in enumerate(zip(S, Y)):
            P, n = int(y.sum()), y.size
            if P > 0:
                ym = y.copy()
                ym[y] = rng.random(P) > a
            elif rng.random() < b:
                ym = rng.random(n) < d
            else:
                ym = np.zeros(n, bool)
            per_change[i] = (
                metric(S[i], ym)
                if metric.structure != "label"
                else score(metric, S[i], thresholds[i], ym)
            )
        agg[m] = per_change.mean(axis=0)
    lo, hi = np.quantile(agg, [0.025, 0.975], axis=0)
    kappa = hi - lo
    return level, kappa, kappa.max()

In [216]:
import plotly.graph_objects as go

rng = np.random.default_rng()
C = changes(300, rng)
n_max = max(n for p, n in C if len(p) > 0)
mu = np.linspace(0.0, np.sqrt(2 * np.log(n_max)) + 2.5, 50)


level, kappa, theta = theta_star(
    C, mu, metric=f1, alpha=(2, 18), beta=(7, 3), delta=(4, 4), M=500, rng=rng
)

o = np.argsort(level)  # nach Level sortieren -> x monoton

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=level[o],
        y=kappa[o],
        mode="lines+markers",
        name="κ(level)",
        line=dict(color="steelblue"),
        marker=dict(size=6),
    )
)
fig.add_hline(
    y=theta,
    line_dash="dash",
    line_color="crimson",
    annotation_text=f"θ* = max κ = {theta:.3f}",
    annotation_position="top left",
)
fig.update_layout(
    title="Auflösungs-Topographie — θ* = worst-case κ",
    xaxis_title="aggregierter sauberer F1 (Qualitäts-Level)",
    yaxis_title="κ  (Breite des 95%-Bands unter Labelrauschen)",
    template="simple_white",
)
fig.show()

In [226]:
import numpy as np
import plotly.graph_objects as go
from math import erf

# assumes changes, f1, mcc, mrr, theta_star are in scope

ALPHA = (2, 18)  # false discovery on positive nodes      (mean 0.10)
BETA = (2, 38)  # change defective among negative labels  (mean 0.05)
DELTA = (4, 6)  # defect density in false negatives       (mean 0.40)

M = 300
GRID = np.linspace(0.0, 4.0, 40)
auroc = 0.5 * (1.0 + np.array([erf(d / 2.0) for d in GRID]))  # AUROC = Phi(delta/sqrt2)

C = changes(120, rng=np.random.default_rng(0))

fig = go.Figure()
for metric, name, col in [
    (f1, "F1", "#185FA5"),
    (mcc, "MCC", "#1D9E75"),
    (mrr, "MRR", "#D85A30"),
]:
    level, kappa, kstar = theta_star(
        C,
        GRID,
        metric=metric,
        M=M,
        alpha=ALPHA,
        beta=BETA,
        delta=DELTA,
        rng=np.random.default_rng(1),
    )
    j = int(np.argmax(kappa))
    fig.add_trace(
        go.Scatter(
            x=auroc, y=kappa, mode="lines", name=name, line=dict(color=col, width=2)
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[auroc[j]],
            y=[kstar],
            mode="markers",
            marker=dict(color=col, size=9),
            showlegend=False,
        )
    )

fig.update_layout(
    xaxis_title="Φ(δ/√2)",
    yaxis_title="κ(δ)",
    template="simple_white",
    width=760,
    height=460,
    legend=dict(x=0.01, y=0.99),
)
fig.show()
# fig.write_image("third/resolution-curve.pdf")   # needs kaleido

In [245]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from math import erf

ALPHA, BETA, DELTA = (2, 18), (2, 38), (4, 6)
M = 300
GRID = np.linspace(0.0, 8.0, 60)
auroc = 0.5 * (1.0 + np.array([erf(d / 2.0) for d in GRID]))

C_full = changes(120, rng=np.random.default_rng(0))
C_rank = [(p, n) for p, n in C_full if len(p) > 0]  # rank metrics: defect-bearing only

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09)
for metric, name, col in [
    (f1, "F1", "#0072B2"),  # Blue
    # (mcc, "MCC", "#009E73"), # Bluish Green
    (mrr, "MRR", "#E69F00"),  # Vermillion
]:
    C = C_full if metric.structure == "label" else C_rank
    level, kappa, kstar = theta_star(
        C,
        GRID,
        metric=metric,
        M=M,
        alpha=ALPHA,
        beta=BETA,
        delta=DELTA,
        rng=np.random.default_rng(1),
    )
    fig.add_trace(
        go.Scatter(x=auroc, y=level, name=name, line=dict(color=col, width=2)),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=auroc, y=kappa, line=dict(color=col, width=2), showlegend=False),
        row=2,
        col=1,
    )

fig.update_xaxes(title_text="Φ(δ/√2)", row=2, col=1)
fig.update_yaxes(title_text="metric", row=1, col=1)
fig.update_yaxes(title_text="κ(δ)", row=2, col=1)
fig.update_layout(
    template="simple_white",
    width=800,
    height=400,
    margin=dict(t=30, b=45, l=65, r=20),
    font=dict(family="IBM Plex Sans, sans-serif"),
    # legend=dict(x=0.01, y=0.99),
)
fig.show(
    config={
        "toImageButtonOptions": {
            "format": "svg",
            "filename": "plot",
            "width": 800,
            "height": 400,
            "scale": 1,
        }
    }
)